# Session 2: LangChain Development & Tool Integration (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 2. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks. Use this as your reference during class delivery.

## Learning Objectives

By the end of this session, students will be able to:

1. **Use ChatModels and PromptTemplates** as LangChain's core building blocks
2. **Compose chains with LCEL** using the pipe operator (`|`)
3. **Create custom tools** using the `@tool` decorator
4. **Load and split documents** for retrieval workflows
5. **Build a RAG pipeline** that grounds LLM answers in external knowledge
6. **Manage conversation memory** across multi-turn interactions

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: LangChain Basics — ChatModels and PromptTemplates

ChatModels wrap LLM APIs into a unified interface. PromptTemplates let you parameterize prompts with variables so they become reusable across different inputs.

In [ ]:
# Demo 1 - LangChain Basics: ChatModels and PromptTemplates

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

response = llm.invoke("What is LangChain in one sentence?")
print("Simple invocation:")
print(f"  Type: {type(response)}")
print(f"  Content: {response.content}")

messages = [
    SystemMessage(content="You are a Python expert. Be concise."),
    HumanMessage(content="What is a list comprehension?")
]
response = llm.invoke(messages)
print(f"\nWith messages:\n  {response.content}")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Answer in {style} style."),
    ("human", "{question}")
])

formatted = prompt.format_messages(
    domain="machine learning", style="casual", question="What is gradient descent?"
)
print(f"\nFormatted prompt messages: {len(formatted)}")
for msg in formatted:
    print(f"  [{msg.type}]: {msg.content[:80]}...")

response = llm.invoke(formatted)
print(f"\nResponse: {response.content[:200]}...")

### Demo 2: Building Chains with LCEL (Pipe Operator)

LCEL uses the `|` operator to compose components into chains. The pattern is: `prompt | model | parser`.

In [ ]:
# Demo 2 - Building Chains with LCEL

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains concepts clearly."),
    ("human", "Explain {concept} in exactly {num_sentences} sentences.")
])
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"concept": "recursion", "num_sentences": "3"})
print("Simple chain result:")
print(result)

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output a JSON object with keys: definition, example, difficulty (1-10)."),
    ("human", "Describe the programming concept: {concept}")
])
json_chain = json_prompt | llm | JsonOutputParser()
result = json_chain.invoke({"concept": "decorators"})
print(f"\nJSON chain result (type={type(result).__name__}):")
print(json.dumps(result, indent=2))

summarize_prompt = ChatPromptTemplate.from_template("Summarize this in one sentence: {text}")
translate_prompt = ChatPromptTemplate.from_template("Translate this to French: {text}")
summary_chain = summarize_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | llm | StrOutputParser()

text = "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed."
summary = summary_chain.invoke({"text": text})
print(f"\nSummary: {summary}")
translation = translate_chain.invoke({"text": summary})
print(f"French: {translation}")

### Demo 3: Creating and Using Custom Tools

Tools let LLMs interact with the real world. Use the `@tool` decorator to turn any Python function into a tool.

In [ ]:
# Demo 3 - Creating and Using Custom Tools

@tool
def word_count(text: str) -> int:
    """Count the number of words in a text string."""
    return len(text.split())

@tool
def reverse_string(text: str) -> str:
    """Reverse a given text string."""
    return text[::-1]

@tool
def calculate_average(numbers: list[float]) -> float:
    """Calculate the average of a list of numbers."""
    if not numbers:
        return 0.0
    return sum(numbers) / len(numbers)

print("Tool: word_count")
print(f"  Name: {word_count.name}")
print(f"  Description: {word_count.description}")

print(f"\nword_count('Hello world from LangChain'): {word_count.invoke('Hello world from LangChain')}")
print(f"reverse_string('LangChain'): {reverse_string.invoke('LangChain')}")
print(f"calculate_average([10, 20, 30]): {calculate_average.invoke([10, 20, 30])}")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools([word_count, reverse_string, calculate_average])
response = llm_with_tools.invoke("How many words are in the sentence 'The quick brown fox jumps'?")
print(f"\nLLM response with tools:")
print(f"  Content: {response.content}")
print(f"  Tool calls: {response.tool_calls}")

### Demo 4: Document Loading and Text Splitting

RAG starts with loading documents and splitting them into manageable chunks.

In [ ]:
# Demo 4 - Document Loading and Text Splitting

documents = [
    Document(
        page_content="""Large Language Models (LLMs) are neural networks trained on massive text datasets.
They use the Transformer architecture, which relies on self-attention mechanisms to process
sequences of tokens. Modern LLMs like GPT-4 and Claude have billions of parameters.""",
        metadata={"source": "llm_overview.txt", "chapter": 1}
    ),
    Document(
        page_content="""Retrieval-Augmented Generation (RAG) combines LLMs with external knowledge retrieval.
Instead of relying solely on training data, RAG retrieves relevant documents and includes
them in the prompt context. This reduces hallucinations and keeps information up-to-date.""",
        metadata={"source": "rag_guide.txt", "chapter": 2}
    )
]

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  Source: {doc.metadata['source']} | Length: {len(doc.page_content)} chars")

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30, separators=["\n\n", "\n", ". ", " ", ""])
chunks = splitter.split_documents(documents)
print(f"\nSplit into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1} ({len(chunk.page_content)} chars) [{chunk.metadata['source']}]: {chunk.page_content[:80]}...")

### Demo 5: Building a Simple RAG Pipeline

Knowledge base + retrieval + LCEL chain = a grounded Q&A system.

In [ ]:
# Demo 5 - Building a Simple RAG Pipeline

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

knowledge_base = [
    "LangChain is a framework for building LLM applications. It provides composable components for prompts, models, and chains.",
    "LCEL (LangChain Expression Language) uses the pipe operator (|) to chain components. The pattern is: prompt | model | parser.",
    "LangGraph extends LangChain with graph-based workflows. It supports cycles, conditional edges, and persistent state.",
    "RAG (Retrieval-Augmented Generation) improves LLM accuracy by retrieving relevant documents before generating answers.",
    "Tools in LangChain are defined using the @tool decorator. They allow LLMs to interact with external systems and APIs.",
    "Memory in LangChain stores conversation history. Common types include ConversationBufferMemory and ConversationSummaryMemory."
]

def simple_retrieve(query, k=2):
    scored = []
    query_words = set(query.lower().split())
    for chunk in knowledge_base:
        chunk_words = set(chunk.lower().split())
        overlap = len(query_words & chunk_words)
        scored.append((overlap, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [chunk for _, chunk in scored[:k]]

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based ONLY on the context. If not found, say 'I don't have enough information.'\n\nContext:\n{context}"),
    ("human", "{question}")
])
rag_chain = rag_prompt | llm | StrOutputParser()

for question in ["What is LCEL?", "How does RAG work?", "What is quantum computing?"]:
    retrieved = simple_retrieve(question)
    context = "\n".join(f"- {c}" for c in retrieved)
    answer = rag_chain.invoke({"context": context, "question": question})
    print(f"Q: {question}\nA: {answer}\n")

---
## Tasks -- Full Solutions

Below are the complete, working solutions for all four student tasks.

### Task 1: Build a Chain with Prompt Templates and Output Parsers

**Approach:** We create a ChatPromptTemplate that instructs the model to output JSON with the quiz question structure. We chain it with JsonOutputParser for automatic parsing. The key insight is that the system message defines the output schema.

In [ ]:
# Task 1 - SOLUTION: Build a Chain with Prompt Templates and Output Parsers

def create_quiz_chain():
    """Create a chain that generates quiz questions."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are a quiz generator. Output a JSON object with these fields: "
            "question (string), options (list of 4 strings), correct_answer (string matching one option). "
            "Generate questions appropriate for the given difficulty level."
        )),
        ("human", "Generate a {difficulty} quiz question about {topic}.")
    ])

    chain = prompt | llm | JsonOutputParser()
    return chain


# Test
chain = create_quiz_chain()
result = chain.invoke({"topic": "Python programming", "difficulty": "intermediate"})
print(json.dumps(result, indent=2))

### Task 2: Create Custom Tools for Math and Text Processing

**Approach:** Each tool is a decorated Python function with a clear docstring. The Fibonacci tool uses iterative generation, text_stats counts words/chars/sentences, and caesar_cipher shifts ASCII letters. We then bind all three to an LLM.

In [ ]:
# Task 2 - SOLUTION: Create Custom Tools for Math and Text Processing

@tool
def fibonacci(n: int) -> list:
    """Return the first n Fibonacci numbers as a list."""
    if n <= 0:
        return []
    if n == 1:
        return [0]
    fibs = [0, 1]
    for _ in range(2, n):
        fibs.append(fibs[-1] + fibs[-2])
    return fibs

@tool
def text_stats(text: str) -> dict:
    """Return word count, character count, and sentence count for a text."""
    words = len(text.split())
    chars = len(text)
    sentences = text.count('.') + text.count('!') + text.count('?')
    return {"word_count": words, "char_count": chars, "sentence_count": max(sentences, 1)}

@tool
def caesar_cipher(text: str, shift: int) -> str:
    """Encrypt text using Caesar cipher with the given shift value."""
    result = []
    for char in text:
        if char.isalpha():
            base = ord('A') if char.isupper() else ord('a')
            result.append(chr((ord(char) - base + shift) % 26 + base))
        else:
            result.append(char)
    return ''.join(result)


# Test tools directly
print(f"fibonacci(8): {fibonacci.invoke(8)}")
print(f"text_stats: {text_stats.invoke('Hello world. How are you?')}")
print(f"caesar_cipher: {caesar_cipher.invoke({'text': 'hello', 'shift': 3})}")

# Bind to LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools([fibonacci, text_stats, caesar_cipher])
response = llm_with_tools.invoke("Give me the first 10 Fibonacci numbers")
print(f"\nLLM tool calls: {response.tool_calls}")

### Task 3: Implement a Conversational Chain with Memory

**Approach:** The `ConversationalChain` class uses a `MessagesPlaceholder` in the prompt to inject chat history. Each call appends HumanMessage and AIMessage objects to the history list, giving the LLM full context of the conversation.

In [ ]:
# Task 3 - SOLUTION: Implement a Conversational Chain with Memory

class ConversationalChain:
    def __init__(self):
        """Initialize the chain with memory."""
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful assistant. Remember the conversation history."),
            MessagesPlaceholder("history"),
            ("human", "{input}")
        ])
        self.chain = self.prompt | self.llm | StrOutputParser()
        self.history = []

    def chat(self, user_input):
        """Send a message and get a response, maintaining history."""
        response = self.chain.invoke({
            "history": self.history,
            "input": user_input
        })
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=response))
        return response

    def get_history(self):
        """Return the conversation history."""
        return self.history


# Test
conv = ConversationalChain()
print("USER: My name is Alex. I'm learning LangChain.")
print(f"AGENT: {conv.chat('My name is Alex. I am learning LangChain.')}")
print()
print("USER: What's my name and what am I learning?")
print(f"AGENT: {conv.chat('What is my name and what am I learning?')}")
print(f"\nHistory: {len(conv.get_history())} messages")

### Task 4: Build a RAG-Powered Q&A System

**Approach:** The `SimpleRAG` class accepts Document objects, splits them with `RecursiveCharacterTextSplitter`, and stores the chunks. The `retrieve` method uses keyword overlap scoring. The `ask` method retrieves context, includes source metadata, and uses an LCEL chain to generate an answer.

In [ ]:
# Task 4 - SOLUTION: Build a RAG-Powered Q&A System

class SimpleRAG:
    def __init__(self, documents):
        """Initialize with a list of Document objects, split into chunks."""
        splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
        self.chunks = splitter.split_documents(documents)
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", (
                "Answer the question based ONLY on the provided context. "
                "Include which source(s) you used. "
                "If the context does not contain the answer, say so.\n\n"
                "Context:\n{context}"
            )),
            ("human", "{question}")
        ])
        self.chain = self.prompt | self.llm | StrOutputParser()
        print(f"Initialized SimpleRAG with {len(self.chunks)} chunks from {len(documents)} documents")

    def retrieve(self, query, k=3):
        """Retrieve the top-k most relevant chunks for a query."""
        scored = []
        query_words = set(query.lower().split())
        for chunk in self.chunks:
            chunk_words = set(chunk.page_content.lower().split())
            score = len(query_words & chunk_words)
            scored.append((score, chunk))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [chunk for _, chunk in scored[:k]]

    def ask(self, question):
        """Answer a question using retrieved context."""
        retrieved = self.retrieve(question)
        context_parts = []
        for chunk in retrieved:
            source = chunk.metadata.get('source', 'unknown')
            context_parts.append(f"[Source: {source}] {chunk.page_content}")
        context = "\n\n".join(context_parts)
        return self.chain.invoke({"context": context, "question": question})


# Test
docs = [
    Document(
        page_content="Python was created by Guido van Rossum and first released in 1991. It emphasizes code readability with significant whitespace. Python supports multiple programming paradigms.",
        metadata={"source": "python.txt"}
    ),
    Document(
        page_content="LangChain provides composable components for building LLM applications. It includes ChatModels, PromptTemplates, OutputParsers, and the LCEL pipe operator for chaining.",
        metadata={"source": "langchain.txt"}
    ),
]

rag = SimpleRAG(docs)
print(rag.ask("Who created Python?"))
print()
print(rag.ask("What components does LangChain provide?"))

---
## Summary

In this session students learned the core LangChain building blocks:

1. **ChatModels & PromptTemplates** -- How LangChain wraps LLM APIs and provides reusable, parameterized prompt templates.
2. **LCEL Chains** -- How the pipe operator (`|`) composes prompt, model, and parser into clean, readable chains.
3. **Custom Tools** -- How the `@tool` decorator turns Python functions into tools that LLMs can discover and invoke.
4. **Document Loading & Splitting** -- How to prepare text for retrieval by splitting it into overlapping chunks.
5. **RAG Pipelines** -- How to combine retrieval with generation to ground LLM answers in external knowledge.

**Instructor Notes:**
- Emphasize LCEL as the modern LangChain way -- discourage legacy `LLMChain` usage.
- For Task 2, show how the docstring becomes the tool description the LLM sees.
- For Task 3, discuss what happens when history grows beyond the context window.
- For Task 4, note that keyword matching is a simplification -- production uses vector similarity.

**Up Next:** In Session 3, we will move from linear chains to graph-based workflows with LangGraph.